In [9]:
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

Matplotlib is building the font cache; this may take a moment.


In [3]:
@dataclass
class Pin:
    local_pos: np.ndarray
    net: str

@dataclass
class Component:
    id: str
    pos: np.ndarray
    theta: float = 0.0
    half_size: np.ndarray = field(default_factory=lambda: np.array([2.5, 2.5]))
    pins: list = field(default_factory=list)

def rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def pin_world(component, pin):
    return component.pos + rot(component.theta) @ pin.local_pos

In [6]:
board = np.array([[0,0], [30, 0], [30, 20], [15, 20], [15, 10], [0, 10]], float)

components = {
    "U1": Component("U1", np.array([10.0, 5.0]), half_size=np.array([3.0, 3.0]),
                    pins=[Pin(np.array([-2.5, 2.5]), "VCC"),
                          Pin(np.array([-2.5, -2.5]), "GND"),
                          Pin(np.array([2.5, 0.0]), "SIG")]),
    "R1": Component("R1", np.array([20.0, 5.0]), theta=np.pi/2,
                    half_size=np.array([1.0, 0.4]),
                    pins=[Pin(np.array([-1.0, 0]), "SIG"),
                          Pin(np.array([1.0, 0]), "GND")]),
}

In [ ]:
def draw(components, board):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.set_aspect("equal")
    ax.add_patch(patches.Polygon(board, facecolor="#f6f6e8", edgecolor="#444"))
    for cid, c in components.items():
        box = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]]) * c.half_size
        corners = (rot(c.theta) @ box.T).T + c.pos
        ax.add_patch(patches.Polygon(corners, facecolor="#cce", edgecolor="#225"))
        ax.text(*c.pos, cid, ha="center", va="center", fontsize=8)
        for p in c.pins:
            ax.plot(*pin_world(c, p), "o", color="#225", markersize=3)
    ax.autos